# Protein–ligand interactions from a single structure (ProLIF)

Computes a ProLIF interaction fingerprint from a **single structure** (one frame,
no trajectory) using `openfe_analysis.prolif.ProLIFAnalysis`, shows the 2D
ligand-interaction network and the 3D view, and covers **water-mediated
interactions** (WaterBridge) for a water model MDAnalysis does not recognise.

The example is `ox2_solvated_complex.pdb` from Zenodo: a membrane-embedded
protein–ligand complex (POPC lipids, SPC water, ligand `UNK`).

**Environment** — see `prolif_from_trajectory.ipynb` for the conda-forge setup;
`py3Dmol` is required for the 3D view.

In [ ]:
import pooch
import MDAnalysis as mda

from openfe_analysis.prolif import ProLIFAnalysis
from openfe_analysis.utils.plotting import plot_prolif_lignetwork, plot_prolif_3d

## 1. Download and load the structure

A single PDB gives a `Universe` with one frame — no `FEReader` or trajectory needed.
The file is fetched from Zenodo record
[10.5281/zenodo.22828040](https://doi.org/10.5281/zenodo.22828040) and cached.

In [ ]:
fetcher = pooch.create(
    path=pooch.os_cache("openfe_analysis"),
    base_url="doi:10.5281/zenodo.22828040",
    registry={"ox2_solvated_complex.pdb": "md5:1d6f9b377b8a2cb57909f1010c2c4934"},
)
pdb = fetcher.fetch("ox2_solvated_complex.pdb")

u = mda.Universe(pdb)
print("n_atoms :", u.atoms.n_atoms)
print("n_frames:", len(u.trajectory))
print("resnames:", sorted(set(u.residues.resnames)))

## 2. Select the ligand

> If a structure contains several ligands (e.g. SepTop systems with two `UNK`
> residues), select **one** at a time — e.g. `"resname UNK and segid B"`. Selecting
> several at once builds a single malformed ligand molecule.

In [ ]:
ligand_ag = u.select_atoms("resname UNK")
print("ligand atoms:", ligand_ag.n_atoms)

## 3. Run the fingerprint (default interactions)

`interactions=None` uses ProLIF's default set. The structure has a single frame,
so `run()` computes one fingerprint.

In [ ]:
analysis = ProLIFAnalysis(u, ligand_ag)
analysis.run(n_jobs=1, progress=True)

df = analysis.to_dataframe()
print("interactions detected:", df.shape[1])
df

## 4. 2D ligand-interaction network

In [ ]:
plot_prolif_lignetwork(analysis.fp, analysis.ligand_ag)

## 5. 3D interaction view (py3Dmol)

In [ ]:
plot_prolif_3d(
    analysis.fp,
    analysis.ligand_ag,
    analysis.protein_ag,
    frame=0,
)

## 6. Water-mediated interactions (WaterBridge)

WaterBridge needs to know which atoms are water. By default `ProLIFAnalysis`
uses MDAnalysis's `"water"` keyword, which recognises common resnames (`HOH`,
`WAT`, `SOL`, `TIP3`, ...) but **not** `SPC` — so in this system it finds no water:

In [ ]:
print('"water"       :', u.select_atoms("water").n_atoms, "atoms")
print('"resname SPC" :', u.select_atoms("resname SPC").n_atoms, "atoms")

Pass `water_selection` to point `ProLIFAnalysis` at the right atoms. Bonds are
then also guessed on the waters (needed to detect the water O–H donors), which
takes a little while for ~7,900 SPC molecules.

> Without `water_selection`, WaterBridge would be dropped with a warning because
> no water is found (and an error is raised if it was the only interaction).

In [ ]:
wb = ProLIFAnalysis(
    u,
    ligand_ag,
    interactions=["WaterBridge", "HBDonor", "HBAcceptor"],
    water_selection="resname SPC",
)
wb.run(n_jobs=1, progress=True)
wb.to_dataframe()

In [ ]:
plot_prolif_lignetwork(wb.fp, wb.ligand_ag)

The 3D view now includes the pocket waters taking part in the bridges:

In [ ]:
plot_prolif_3d(
    wb.fp,
    wb.ligand_ag,
    wb.protein_ag,
    wb.water_ag,
    frame=0,
)